In [1]:

import torch

if torch.cuda.is_available():
    print("GPU is available!")
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available. We need to check the runtime settings.")


GPU is available!
GPU name: Tesla T4


In [2]:

import json
from pathlib import Path

chunks_path = Path("/content/chunks.json")

if chunks_path.exists():
    with open(chunks_path, "r", encoding="utf-8") as file:
        chunks = json.load(file)

    print("ISM chunks uploaded successfully!")
    print("Total chunks:", len(chunks))
    print("First chunk unit:", chunks[0]["unit"])
    print("First chunk page:", chunks[0]["page_number"])
else:
    print("chunks.json not found. Please upload it first.")


ISM chunks uploaded successfully!
Total chunks: 398
First chunk unit: Unit1
First chunk page: 1


In [3]:

import json
from pathlib import Path

chunks_path = Path("/content/chunks.json")

with open(chunks_path, "r", encoding="utf-8") as file:
    chunks = json.load(file)

print("TOTAL CHUNKS:", len(chunks))
print("\nAVAILABLE UNITS:")

units = sorted(set(chunk["unit"] for chunk in chunks))
for unit in units:
    count = sum(1 for chunk in chunks if chunk["unit"] == unit)
    print(f"{unit}: {count} chunks")

print("\nSAMPLE CONTENT:")
for chunk in chunks[:3]:
    print("\n" + "=" * 50)
    print("Unit:", chunk["unit"])
    print("PDF:", chunk["source_file"])
    print("Page:", chunk["page_number"])
    print("Text:", chunk["text"][:500])


TOTAL CHUNKS: 398

AVAILABLE UNITS:
Unit1: 101 chunks
Unit2: 55 chunks
Unit3: 169 chunks
Unit4: 73 chunks

SAMPLE CONTENT:

Unit: Unit1
PDF: Unit1.pdf
Page: 1
Text: 18CSE360T INFORMATION STORAGE AND MANAGEMENT SRM INSTITUTE OF SCIENCE ANDTECHNOLOGY, CHENNAI.

Unit: Unit1
PDF: Unit1.pdf
Page: 3
Text: CLR AND CLO

Unit: Unit1
PDF: Unit1.pdf
Page: 4
Text: CO-PO-PSO Mapping


In [4]:

import json
from pathlib import Path

INPUT_FILE = Path("/content/chunks.json")
OUTPUT_FILE = Path("/content/ism_training_draft.jsonl")

with open(INPUT_FILE, "r", encoding="utf-8") as file:
    chunks = json.load(file)

training_examples = []

# Create draft question-answer examples from source passages.
# These are retrieval-grounded drafts, not verified exam answers.
for chunk in chunks:
    text = chunk.get("text", "").strip()

    if len(text.split()) < 40:
        continue

    unit = chunk["unit"]
    source_file = chunk["source_file"]
    page = chunk["page_number"]

    question = (
        f"Explain the important concepts covered in this passage "
        f"from {unit} for exam preparation."
    )

    answer = (
        f"**Source:** {source_file}, page {page}\n\n"
        f"**Explanation:**\n{text}\n\n"
        "This draft answer is based on the cited passage. "
        "Expand it with other relevant syllabus content as needed."
    )

    training_examples.append({
        "unit": unit,
        "source_file": source_file,
        "page_number": page,
        "question": question,
        "answer": answer,
        "verified": False
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
    for example in training_examples:
        file.write(json.dumps(example, ensure_ascii=False) + "\n")

print("Draft training dataset created!")
print("Draft examples:", len(training_examples))
print("Output file:", OUTPUT_FILE)
print("Review required before fine-tuning: YES")

if training_examples:
    print("\nFirst draft example:")
    print(json.dumps(training_examples[0], indent=2, ensure_ascii=False))


Draft training dataset created!
Draft examples: 331
Output file: /content/ism_training_draft.jsonl
Review required before fine-tuning: YES

First draft example:
{
  "unit": "Unit1",
  "source_file": "Unit1.pdf",
  "page_number": 6,
  "question": "Explain the important concepts covered in this passage from Unit1 for exam preparation.",
  "answer": "**Source:** Unit1.pdf, page 6\n\n**Explanation:**\nIntroduction to Information Storage Management, Evolution of Storage Architecture, Data Centre Infrastructure, Virtualization and Cloud Computing, Key challenges in managing Information, Data Center Environment: Application, Database Management System (DBMS), Host : Connectivity, Storage, Disk Drive Components, Disk Drive Performance, Intelligent Storage System, Components of an Intelligent Storage System, Storage Provisioning, Types of Intelligent Storage Systems, Creation of Virtual storage machine, Navigation of storage system . UNIT – I\n\nThis draft answer is based on the cited passage. 

In [5]:

import json
from pathlib import Path

OUTPUT_FILE = Path("/content/ism_exam_format_examples.jsonl")

examples = [
    {
        "question": "Explain a topic from Information Storage Technology Management for 5 marks.",
        "marks": 5,
        "answer_instructions": (
            "Give a concise exam-style answer with a definition, "
            "key points, and a short example if supported by the notes."
        )
    },
    {
        "question": "Explain a topic from Information Storage Technology Management for 10 marks.",
        "marks": 10,
        "answer_instructions": (
            "Give a structured exam-style answer with an introduction, "
            "relevant subheadings, detailed points, and a conclusion."
        )
    },
    {
        "question": "Explain a topic from Information Storage Technology Management for 15 marks.",
        "marks": 15,
        "answer_instructions": (
            "Give a comprehensive exam-style answer with an introduction, "
            "detailed explanations, relevant classifications or examples, "
            "and a conclusion. Do not invent unsupported facts."
        )
    },
    {
        "question": "Explain a topic from Information Storage Technology Management for 20 marks.",
        "marks": 20,
        "answer_instructions": (
            "Give an extensive, well-organized exam-style answer with "
            "relevant headings, detailed explanations, examples or diagrams "
            "where supported, and a conclusion. Do not invent unsupported facts."
        )
    }
]

with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
    for example in examples:
        file.write(json.dumps(example, ensure_ascii=False) + "\n")

print("Exam-format examples created:", len(examples))
print("Saved to:", OUTPUT_FILE)
print("Note: These are formatting instructions, not subject-answer training pairs.")


Exam-format examples created: 4
Saved to: /content/ism_exam_format_examples.jsonl
Note: These are formatting instructions, not subject-answer training pairs.


In [6]:

!nvidia-smi



Sun Oct 18 18:47:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P8             15W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:

%pip install -q transformers datasets peft accelerate trl


In [8]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

print("Model loaded successfully!")
print("GPU available:", torch.cuda.is_available())
print("Model:", MODEL_NAME)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded successfully!
GPU available: True
Model: Qwen/Qwen2.5-1.5B-Instruct


In [9]:
!nvidia-smi

Sun Oct 18 18:48:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P0             30W /   70W |    3151MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
%pip install -q transformers datasets peft accelerate trl


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))
print("Parameters:", f"{model.num_parameters():,}")

Loading tokenizer...
Loading model...
Model loaded successfully!
GPU: Tesla T4
Parameters: 1,543,714,304


In [12]:
import json
import os

# Locate your existing chunks file
possible_paths = [
    "/content/chunks.json",
    "/content/processed_data/chunks.json"
]

chunks_path = next(
    (p for p in possible_paths if os.path.exists(p)),
    None
)

if chunks_path is None:
    raise FileNotFoundError(
        "chunks.json not found. Check the Files panel in Colab."
    )

with open(chunks_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("File loaded:", chunks_path)
print("Number of chunks:", len(chunks))
print("\nFirst chunk preview:")
print(str(chunks[0])[:1000])

File loaded: /content/chunks.json
Number of chunks: 398

First chunk preview:
{'chunk_id': 1, 'unit': 'Unit1', 'source_file': 'Unit1.pdf', 'page_number': 1, 'text': '18CSE360T INFORMATION STORAGE AND MANAGEMENT SRM INSTITUTE OF SCIENCE ANDTECHNOLOGY, CHENNAI.'}


In [13]:
from collections import Counter

print("Chunks per unit:")
unit_counts = Counter(chunk["unit"] for chunk in chunks)

for unit, count in sorted(unit_counts.items()):
    print(f"{unit}: {count} chunks")

print("\nSample passage from each unit:")
for unit in sorted(unit_counts):
    sample = next(c for c in chunks if c["unit"] == unit)
    print(f"\n--- {unit} | Page {sample['page_number']} ---")
    print(sample["text"][:500])

Chunks per unit:
Unit1: 101 chunks
Unit2: 55 chunks
Unit3: 169 chunks
Unit4: 73 chunks

Sample passage from each unit:

--- Unit1 | Page 1 ---
18CSE360T INFORMATION STORAGE AND MANAGEMENT SRM INSTITUTE OF SCIENCE ANDTECHNOLOGY, CHENNAI.

--- Unit2 | Page 1 ---
18CSE360T INFORMATION STORAGE AND MANAGEMENT SRM INSTITUTE OF SCIENCE AND TECHNOLOGY, CHENNAI.

--- Unit3 | Page 1 ---
Unit - III Business Continuity and Backup Recovery

--- Unit4 | Page 1 ---
Unit IV Introduction to Storage Security, Recovery and Acquisition


In [14]:
import json
import os

training_examples = []

for chunk in chunks:
    text = chunk.get("text", "").strip()

    # Skip very short passages
    if len(text.split()) < 40:
        continue

    unit = chunk.get("unit", "Unknown")
    page = chunk.get("page_number", "Unknown")

    example = {
        "instruction": (
            f"Explain the important concepts from {unit} "
            "in a clear, student-friendly, exam-oriented way. "
            "Use a definition, key points, and an example where relevant. "
            "Do not introduce unsupported facts."
        ),
        "input": text,
        "output": (
            "Study notes based on the provided passage:\n\n"
            + text
            + f"\n\nSource: {unit}, PDF page {page}."
        ),
        "unit": unit,
        "page_number": page
    }

    training_examples.append(example)

output_path = "/content/ism_training_draft.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for example in training_examples:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

print("Draft examples created:", len(training_examples))
print("Saved to:", output_path)
print("\nFirst example:")
print(json.dumps(training_examples[0], indent=2, ensure_ascii=False)[:1500])

Draft examples created: 331
Saved to: /content/ism_training_draft.jsonl

First example:
{
  "instruction": "Explain the important concepts from Unit1 in a clear, student-friendly, exam-oriented way. Use a definition, key points, and an example where relevant. Do not introduce unsupported facts.",
  "input": "Introduction to Information Storage Management, Evolution of Storage Architecture, Data Centre Infrastructure, Virtualization and Cloud Computing, Key challenges in managing Information, Data Center Environment: Application, Database Management System (DBMS), Host : Connectivity, Storage, Disk Drive Components, Disk Drive Performance, Intelligent Storage System, Components of an Intelligent Storage System, Storage Provisioning, Types of Intelligent Storage Systems, Creation of Virtual storage machine, Navigation of storage system . UNIT – I",
  "output": "Study notes based on the provided passage:\n\nIntroduction to Information Storage Management, Evolution of Storage Architecture,

In [15]:
import json

exam_examples = [
    {
        "question": "What is a backup?",
        "marks": 2,
        "answer": (
            "A backup is a copy of data maintained so that the original "
            "data can be restored if it is lost, damaged, or corrupted."
        )
    },
    {
        "question": "Explain the purpose of backup.",
        "marks": 5,
        "answer": (
            "1. Data protection: protects against accidental deletion.\n"
            "2. Recovery: helps restore lost or corrupted data.\n"
            "3. Business continuity: reduces disruption after failures.\n"
            "4. Disaster recovery: supports recovery after major incidents.\n"
            "5. Data preservation: retains copies for future use."
        )
    },
    {
        "question": "Explain backup methods.",
        "marks": 10,
        "answer": (
            "Introduction: Backup methods describe how data is copied "
            "for recovery.\n\n"
            "1. Full backup: copies all selected data.\n"
            "2. Incremental backup: copies data changed since the last backup.\n"
            "3. Differential backup: copies data changed since the last full backup.\n\n"
            "Conclusion: The appropriate method depends on storage capacity, "
            "backup time, and recovery requirements."
        )
    },
    {
        "question": "Explain business continuity planning.",
        "marks": 15,
        "answer": (
            "Introduction: Business continuity planning prepares an "
            "organization to continue critical operations during disruption.\n\n"
            "1. Identify critical business processes.\n"
            "2. Conduct a business impact analysis.\n"
            "3. Analyze risks and possible failures.\n"
            "4. Define recovery objectives.\n"
            "5. Develop continuity and recovery strategies.\n"
            "6. Document roles and responsibilities.\n"
            "7. Test the plan through exercises.\n"
            "8. Review and update the plan regularly.\n\n"
            "Conclusion: A tested plan helps reduce downtime and supports "
            "the recovery of critical services."
        )
    },
    {
        "question": "Explain storage security and its major components.",
        "marks": 20,
        "answer": (
            "Introduction: Storage security protects stored information "
            "from unauthorized access, loss, alteration, and disruption.\n\n"
            "1. Identify information assets.\n"
            "2. Identify threats and vulnerabilities.\n"
            "3. Apply authentication and authorization.\n"
            "4. Protect application access.\n"
            "5. Secure management access.\n"
            "6. Protect backup and archive systems.\n"
            "7. Apply encryption where appropriate.\n"
            "8. Monitor access and maintain accountability.\n"
            "9. Test recovery procedures.\n"
            "10. Review controls and address weaknesses.\n\n"
            "Conclusion: Multiple security controls work together to protect "
            "data availability, confidentiality, and integrity."
        )
    }
]

output_path = "/content/ism_exam_format_examples.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for item in exam_examples:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Exam-format examples created:", len(exam_examples))
print("Saved to:", output_path)


Exam-format examples created: 5
Saved to: /content/ism_exam_format_examples.jsonl


In [16]:
from peft import LoraConfig, get_peft_model, TaskType

# Configure LoRA for efficient fine-tuning
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    bias="none"
)

# Attach LoRA adapters to the loaded model
model = get_peft_model(model, lora_config)

# Display trainable parameters
model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [17]:
import torch
import transformers
import peft

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu130
Transformers: 4.56.2
PEFT: 0.17.1
CUDA available: True


In [18]:
%pip install -q --upgrade "transformers==4.51.3" "peft==0.15.2" "accelerate==1.6.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.23.0 requires transformers>=4.56.1, but you have transformers 4.51.3 which is incompatible.


In [19]:
%pip install -q --upgrade \
  "transformers==4.56.2" \
  "peft==0.17.1" \
  "accelerate>=1.1.0" \
  "trl==0.23.0"

In [20]:
import torch
import transformers
import peft
import accelerate
import trl

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("TRL:", trl.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu130
Transformers: 4.56.2
PEFT: 0.17.1
Accelerate: 1.15.0
TRL: 0.23.0
CUDA available: True


In [21]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model onto GPU...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("\nModel loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))
print("Model parameters:", f"{model.num_parameters():,}")
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

Loading tokenizer...
Loading model onto GPU...

Model loaded successfully!
GPU: Tesla T4
Model parameters: 1,543,714,304
GPU memory allocated: 2.88 GB


In [23]:
import json
import re
import torch

# Use the draft passages already loaded from your PDFs
assert "draft_lines" in globals(), (
    "Please rerun the cell that loads ism_training_draft.jsonl."
)

model.eval()
synthetic_examples = []

# Select two passages from each unit
selected = []
for unit in ["Unit1", "Unit2", "Unit3", "Unit4"]:
    unit_items = [
        x for x in draft_lines
        if x.get("unit") == unit
        and len(x.get("input", "").split()) >= 50
    ]
    selected.extend(unit_items[:2])

print("Selected passages:", len(selected))

for i, item in enumerate(selected):
    passage = item["input"][:3500]
    unit = item.get("unit", "Unknown")
    page = item.get("page_number", "Unknown")

    prompt = f"""
You are preparing study material for an Information Storage
Technology Management (ISM) exam.

Use ONLY the source passage below.
Create one useful exam question and a correct answer suitable for
either 5 or 10 marks. Choose the mark count based on the passage.
Do not invent facts or include information absent from the passage.
Return valid JSON only, using exactly these keys:
"question", "marks", "answer".

SOURCE ({unit}, PDF page {page}):
{passage}
"""

    messages = [
        {
            "role": "system",
            "content": "You create accurate, source-grounded exam study material."
        },
        {"role": "user", "content": prompt}
    ]

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted,
        return_tensors="pt",
        truncation=True,
        max_length=1800
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Extract a JSON object if the model wrapped it in extra text
    match = re.search(r"\{.*\}", generated, re.DOTALL)

    if match:
        try:
            result = json.loads(match.group())
            if all(k in result for k in ["question", "marks", "answer"]):
                synthetic_examples.append({
                    "unit": unit,
                    "page_number": page,
                    "question": result["question"],
                    "marks": result["marks"],
                    "answer": result["answer"]
                })
        except json.JSONDecodeError:
            pass

print("\nValid examples generated:", len(synthetic_examples))

for example in synthetic_examples[:3]:
    print("\n" + "=" * 50)
    print(json.dumps(example, indent=2, ensure_ascii=False))

AssertionError: Please rerun the cell that loads ism_training_draft.jsonl.

In [ ]:
import json
import os

draft_path = "/content/ism_training_draft.jsonl"

if os.path.exists(draft_path):
    with open(draft_path, "r", encoding="utf-8") as f:
        draft_lines = [
            json.loads(line)
            for line in f
            if line.strip()
        ]

    print("Training draft loaded successfully!")
    print("Total examples:", len(draft_lines))
    print("Available units:", sorted({
        item.get("unit", "Unknown")
        for item in draft_lines
    }))

else:
    print("Draft file not found.")
    print("Files currently in /content:")
    print(os.listdir("/content"))

In [ ]:
print("First question instruction:")
print(draft_lines[0]["instruction"])

print("\nSource unit:", draft_lines[0]["unit"])
print("Source page:", draft_lines[0]["page_number"])

print("\nSource passage:")
print(draft_lines[0]["input"][:500])

In [ ]:
import json
import torch

model.eval()
synthetic_examples = []

# Select one passage from each unit
selected = []

for unit in ["Unit1", "Unit2", "Unit3", "Unit4"]:
    candidates = [
        item for item in draft_lines
        if item.get("unit") == unit
        and len(item.get("input", "").split()) >= 50
    ]

    if candidates:
        selected.append(candidates[0])

print("Selected passages:", len(selected))

for item in selected:
    unit = item["unit"]
    page = item["page_number"]
    passage = item["input"][:2500]

    messages = [
        {
            "role": "system",
            "content": (
                "You are an ISM exam study assistant. "
                "Use only the provided passage. Do not invent facts."
            )
        },
        {
            "role": "user",
            "content": f"""
Create one exam question and answer based only on this passage.
The answer should suit a 5-mark question.

Return JSON with exactly these keys:
question, marks, answer

Unit: {unit}
PDF page: {page}

Passage:
{passage}
"""
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1400
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    answer_text = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    synthetic_examples.append({
        "unit": unit,
        "page_number": page,
        "source_passage": passage,
        "draft_answer": answer_text
    })

    print(f"\n--- {unit}, page {page} ---")
    print(answer_text[:1000])

print("\nDrafts generated:", len(synthetic_examples))

In [ ]:
for i, example in enumerate(synthetic_examples, start=1):
    print("=" * 60)
    print(f"EXAMPLE {i}")
    print("Unit:", example["unit"])
    print("PDF page:", example["page_number"])

    print("\nSOURCE PASSAGE:")
    print(example["source_passage"][:700])

    print("\nQWEN'S DRAFT ANSWER:")
    print(example["draft_answer"][:1200])
    print()

In [ ]:
import json
import re

clean_examples = []

for item in synthetic_examples:
    raw = item["draft_answer"].strip()

    # Remove Markdown code fences
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)

    try:
        parsed = json.loads(raw)

        clean_examples.append({
            "unit": item["unit"],
            "page_number": item["page_number"],
            "question": parsed.get("question", ""),
            "marks": parsed.get("marks", 5),
            "answer": parsed.get("answer", ""),
            "source_passage": item["source_passage"]
        })

    except json.JSONDecodeError:
        print(
            "Could not parse draft from",
            item["unit"],
            "page",
            item["page_number"]
        )

output_path = "/content/ism_generated_drafts.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for item in clean_examples:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("\nSuccessfully cleaned:", len(clean_examples))
print("Saved to:", output_path)

In [ ]:
import json
import os

file_path = "/content/ism_generated_drafts.jsonl"

if not os.path.exists(file_path):
    print("File not found. Run Step 15 first.")
else:
    with open(file_path, "r", encoding="utf-8") as f:
        examples = [
            json.loads(line)
            for line in f
            if line.strip()
        ]

    print("Total saved examples:", len(examples))

    for i, item in enumerate(examples, start=1):
        print(f"\nExample {i}")
        print("Unit:", item["unit"])
        print("Page:", item["page_number"])
        print("Question:", item["question"])
        print("Marks:", item["marks"])
        print("Answer preview:", item["answer"][:300])

In [ ]:
import json
import os

# Load generated drafts if the variable is missing
draft_path = "/content/ism_generated_drafts.jsonl"

if "clean_examples" not in globals() or not clean_examples:
    if not os.path.exists(draft_path):
        raise FileNotFoundError(
            "ism_generated_drafts.jsonl not found. "
            "Run the earlier draft-generation and cleaning cells first."
        )

    with open(draft_path, "r", encoding="utf-8") as f:
        clean_examples = [
            json.loads(line)
            for line in f if line.strip()
        ]

# Keep only examples with a question and answer
valid_examples = []

for item in clean_examples:
    question = str(item.get("question", "")).strip()
    answer = str(item.get("answer", "")).strip()

    if question and answer:
        valid_examples.append({
            "unit": item.get("unit", "Unknown"),
            "page_number": item.get("page_number", "Unknown"),
            "question": question,
            "answer": answer,
            "source_passage": item.get("source_passage", "")
        })

# Save a reviewable dataset with source information
review_path = "/content/ism_training_review.jsonl"

with open(review_path, "w", encoding="utf-8") as f:
    for item in valid_examples:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

# Build chat-format training examples
chat_path = "/content/ism_lora_training.jsonl"

with open(chat_path, "w", encoding="utf-8") as f:
    for item in valid_examples:
        record = {
            "messages": [
                {
                    "role": "system",
                    "content": (
                        "You are an Information Storage Technology "
                        "Management exam assistant. Answer clearly and "
                        "accurately. Do not invent facts."
                    )
                },
                {
                    "role": "user",
                    "content": (
                        f"Answer this ISM question: {item['question']}\n"
                        f"Use the topic from {item['unit']}, "
                        f"PDF page {item['page_number']}."
                    )
                },
                {
                    "role": "assistant",
                    "content": item["answer"]
                }
            ]
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Valid draft examples:", len(valid_examples))
print("Review dataset:", review_path)
print("Chat-format dataset:", chat_path)

if valid_examples:
    print("\nExample question:", valid_examples[0]["question"])
    print("Example answer:", valid_examples[0]["answer"][:500])
else:
    print("\nNo valid examples found; do not start training yet.")

In [ ]:
import json
import os

path = "/content/ism_lora_training.jsonl"

if not os.path.exists(path):
    raise FileNotFoundError(
        "Training file not found. Please rerun Step 17."
    )

with open(path, "r", encoding="utf-8") as f:
    training_data = [
        json.loads(line)
        for line in f if line.strip()
    ]

print("Training examples:", len(training_data))
print("GPU:", __import__("torch").cuda.get_device_name(0))

if len(training_data) < 10:
    print(
        "\nWARNING: This is a very small dataset. "
        "It is useful for a pipeline test, but not reliable "
        "for subject-specific fine-tuning."
    )

print("\nFirst training example:")
print(json.dumps(training_data[0], indent=2, ensure_ascii=False)[:1500]
      if training_data else "No examples available.")

In [ ]:
import json
import os
import torch
from datasets import Dataset
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# 1. Load the training dataset
path = "/content/ism_lora_training.jsonl"

if not os.path.exists(path):
    raise FileNotFoundError("Run Step 17 to create the training dataset.")

with open(path, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

if len(records) < 2:
    raise ValueError(
        f"Only {len(records)} examples found. "
        "We need at least 2 to run a training test."
    )

print("Training examples:", len(records))

# 2. Convert chat messages into model text
def format_record(record):
    return {
        "text": tokenizer.apply_chat_template(
            record["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }

dataset = Dataset.from_list(
    [format_record(record) for record in records]
)

# 3. Tokenize examples
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=1024
    )

tokenized_dataset = dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

# 4. Configure training
training_args = TrainingArguments(
    output_dir="/content/ism_lora_output",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    fp16=True,
    report_to="none",
    optim="adamw_torch"
)

# 5. Train the LoRA adapter
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )
)

print("Starting LoRA training...")
trainer.train()

# 6. Save the adapter and tokenizer
save_path = "/content/ism_lora_adapter"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("\nTraining finished!")
print("Adapter saved to:", save_path)

In [ ]:
import torch
import gc
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorForLanguageModeling

# Clean up GPU memory before initiating training
gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

# Ensure only LoRA adapter parameters are trainable
model.train()

training_args = TrainingArguments(
    output_dir="/content/ism_lora_output",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=1e-4,
    logging_steps=1,
    save_strategy="no",
    fp16=False,
    bf16=False,
    report_to="none",
    optim="paged_adamw_8bit",  # Use 8-bit optimizer to drastically reduce memory usage
    max_grad_norm=None
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )
)

# Bypass the gradient scaling check by explicitly setting the scaler to run with allowed fp16 unscaling,
# or telling the accelerator to use a raw scaler wrapper that supports fp16 gradients.
if hasattr(trainer.accelerator, "scaler") and trainer.accelerator.scaler is not None:
    trainer.accelerator.scaler._check_inf_per_device = lambda optimizer: {}
    # Setting allow_fp16 to True inside the inner unscale methods
    original_unscale = trainer.accelerator.scaler._unscale_grads_
    trainer.accelerator.scaler._unscale_grads_ = lambda optimizer, inv_scale, found_inf, allow_fp16=True: original_unscale(optimizer, inv_scale, found_inf, True)

print("Starting training with bypassed FP16 scaling and 8-bit optimizer...")
trainer.train()

save_path = "/content/ism_lora_adapter"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("\nTraining completed!")
print("Adapter saved at:", save_path)

In [ ]:
print("DIAGNOSTIC CELL RAN")

print("Model loaded:", "model" in globals())
print("Trainer exists:", "trainer" in globals())

if "trainer" in globals():
    print("Trainer FP16 setting:", trainer.args.fp16)
    print("Trainer BF16 setting:", trainer.args.bf16)

if "model" in globals():
    print(
        "Trainable parameter types:",
        {
            str(p.dtype)
            for p in model.parameters()
            if p.requires_grad
        }
    )

In [ ]:
# Remove the failed Trainer objects
if "trainer" in globals():
    del trainer

if "training_args" in globals():
    del training_args

import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print("Failed training state cleared.")
print("GPU available:", torch.cuda.is_available())
print("Model still loaded:", "model" in globals())
print("Training dataset available:", "tokenized_dataset" in globals())

In [ ]:
import os
import json
import gc
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType

# 1. Remove old model/trainer references
for name in ["trainer", "model", "base_model", "training_args"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# 2. Load the training data
data_path = "/content/ism_lora_training.jsonl"

if not os.path.exists(data_path):
    raise FileNotFoundError(
        "Training data is missing. We need to recreate the dataset."
    )

with open(data_path, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

if not records:
    raise ValueError("The training dataset is empty.")

print("Training examples:", len(records))
print("GPU:", torch.cuda.get_device_name(0))

# 3. Reload Qwen using FP32
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map={"": 0}
)

# 4. Attach fresh LoRA adapters
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none"
)

model = get_peft_model(base_model, lora_config)
model.config.use_cache = False
model.print_trainable_parameters()

# 5. Format and tokenize the training examples
def format_record(record):
    return tokenizer.apply_chat_template(
        record["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

dataset = Dataset.from_list([
    {"text": format_record(record)}
    for record in records
])

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=512
    )

tokenized_dataset = dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

# 6. Create a fresh Trainer with FP16 and BF16 disabled
training_args = TrainingArguments(
    output_dir="/content/ism_lora_output",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=5e-5,
    logging_steps=1,
    save_strategy="no",
    fp16=False,
    bf16=False,
    report_to="none",
    optim="adamw_torch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )
)

print("\nStarting fresh FP32 training...")
trainer.train()

# 7. Save the LoRA adapter
save_path = "/content/ism_lora_adapter"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("\nTraining completed successfully!")
print("Adapter saved at:", save_path)

In [25]:
import os

print("Checking project files...\n")

for path in [
    "/content/chunks.json",
    "/content/ism_training_draft.jsonl",
    "/content/ism_exam_format_examples.jsonl",
    "/content/ism_generated_drafts.jsonl",
    "/content/ism_training_review.jsonl",
    "/content/ism_lora_training.jsonl",
    "/content/ism_lora_adapter"
]:
    print(f"{'FOUND' if os.path.exists(path) else 'MISSING'}: {path}")

Checking project files...

FOUND: /content/chunks.json
FOUND: /content/ism_training_draft.jsonl
FOUND: /content/ism_exam_format_examples.jsonl
FOUND: /content/ism_generated_drafts.jsonl
FOUND: /content/ism_training_review.jsonl
FOUND: /content/ism_lora_training.jsonl
MISSING: /content/ism_lora_adapter


In [26]:
%pip install -q bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.8 MB/s eta 0:00:00


In [27]:
import os
import json
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)

# Load existing draft training examples
data_path = "/content/ism_lora_training.jsonl"

with open(data_path, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

if not records:
    raise ValueError("Training dataset is empty.")

print("Training examples:", len(records))
print("GPU:", torch.cuda.get_device_name(0))

# Load Qwen in 4-bit precision
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto"
)

base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)

# Attach LoRA adapters
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none"
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# Prepare chat-formatted training data
dataset = Dataset.from_list([
    {
        "text": tokenizer.apply_chat_template(
            item["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }
    for item in records
])

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=512
    )

tokenized_dataset = dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

# Train without mixed-precision gradient scaling
args = TrainingArguments(
    output_dir="/content/ism_lora_output",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=5e-5,
    logging_steps=1,
    save_strategy="no",
    fp16=False,
    bf16=False,
    report_to="none",
    optim="adamw_torch"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )
)

print("Starting 4-bit LoRA training...")
trainer.train()

# Save adapter and tokenizer
save_path = "/content/ism_lora_adapter"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("TRAINING COMPLETED")
print("Adapter saved at:", save_path)

Training examples: 3
GPU: Tesla T4


ImportError: Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`

In [28]:
%pip install -q --upgrade bitsandbytes

In [29]:
import os
import torch
import transformers
import bitsandbytes as bnb

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("BitsAndBytes:", bnb.__version__)
print("GPU:", torch.cuda.get_device_name(0))

print("\nTraining data exists:",
      os.path.exists("/content/ism_lora_training.jsonl"))
print("Chunks file exists:",
      os.path.exists("/content/chunks.json"))

PyTorch: 2.11.0+cu130
Transformers: 4.56.2
BitsAndBytes: 0.50.2
GPU: Tesla T4

Training data exists: True
Chunks file exists: True


In [30]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading Qwen in 4-bit mode...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto"
)

print("\nSUCCESS: Qwen loaded in 4-bit mode!")
print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory used:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

Loading tokenizer...
Loading Qwen in 4-bit mode...


ImportError: Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`

In [31]:
import torch
import bitsandbytes as bnb
from transformers.utils import is_bitsandbytes_available

print("BitsAndBytes version:", bnb.__version__)
print("BitsAndBytes import:", "SUCCESS")
print("CUDA available:", torch.cuda.is_available())
print("Transformers detects BitsAndBytes:", is_bitsandbytes_available())

try:
    print("CUDA device:", torch.cuda.get_device_name(0))
    print("BitsAndBytes CUDA:", bnb.functional.get_special_format_str())
except Exception as e:
    print("Additional check:", repr(e))

BitsAndBytes version: 0.50.2
BitsAndBytes import: SUCCESS
CUDA available: True
Transformers detects BitsAndBytes: False
CUDA device: Tesla T4
Additional check: AttributeError("module 'bitsandbytes.functional' has no attribute 'get_special_format_str'")


In [32]:
import torch
import bitsandbytes as bnb
import transformers

print("PyTorch:", torch.__version__)
print("BitsAndBytes:", bnb.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

try:
    print("GPU:", torch.cuda.get_device_name(0))
    print("BitsAndBytes CUDA check:", bnb.functional.get_special_format_str())
except Exception as e:
    print("Diagnostic error:", repr(e))

from transformers.utils import is_bitsandbytes_available
print("Transformers detects BitsAndBytes:", is_bitsandbytes_available())

PyTorch: 2.11.0+cu130
BitsAndBytes: 0.50.2
Transformers: 4.56.2
CUDA available: True
GPU: Tesla T4
Diagnostic error: AttributeError("module 'bitsandbytes.functional' has no attribute 'get_special_format_str'")
Transformers detects BitsAndBytes: False


In [33]:
%pip install --force-reinstall --no-deps "bitsandbytes==0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 10.2 MB/s eta 0:00:00
  Attempting uninstall: bitsandbytes
    Found existing installation: bitsandbytes 0.50.2
    Uninstalling bitsandbytes-0.50.2:
      Successfully uninstalled bitsandbytes-0.50.2


In [1]:
import torch
import bitsandbytes as bnb
from transformers.utils import is_bitsandbytes_available

print("BitsAndBytes:", bnb.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())
print("Transformers detects BitsAndBytes:", is_bitsandbytes_available())

ERROR:bitsandbytes.cextension:bitsandbytes library load error: Configured CUDA binary not found at /usr/local/lib/python3.13/dist-packages/bitsandbytes/libbitsandbytes_cuda130.so
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/bitsandbytes/cextension.py", line 290, in <module>
    lib = get_native_library()
  File "/usr/local/lib/python3.13/dist-packages/bitsandbytes/cextension.py", line 270, in get_native_library
    raise RuntimeError(f"Configured CUDA binary not found at {cuda_binary_path}")
RuntimeError: Configured CUDA binary not found at /usr/local/lib/python3.13/dist-packages/bitsandbytes/libbitsandbytes_cuda130.so


BitsAndBytes: 0.46.1
GPU: Tesla T4
CUDA available: True
Transformers detects BitsAndBytes: True


In [2]:
import os

print("Chunks:", os.path.exists("/content/chunks.json"))
print("Training data:", os.path.exists("/content/ism_lora_training.jsonl"))

Chunks: True
Training data: True


In [3]:
import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load your PDF chunks
with open("/content/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

# Load the embedding model
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Create embeddings for chunk text
texts = [c.get("text", "") for c in chunks]
embeddings = embedder.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

def retrieve_ism(question, top_k=3):
    query_embedding = embedder.encode(
        [question],
        normalize_embeddings=True
    )
    scores = cosine_similarity(query_embedding, embeddings)[0]
    best_indices = scores.argsort()[-top_k:][::-1]

    results = []
    for idx in best_indices:
        c = chunks[idx]
        results.append({
            "unit": c.get("unit", "Unknown"),
            "page": c.get("page_number", "Unknown"),
            "score": round(float(scores[idx]), 3),
            "text": c.get("text", "")
        })
    return results

# Test a real ISM question
question = "Explain backup granularity"
results = retrieve_ism(question)

for i, result in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Unit:", result["unit"])
    print("PDF page:", result["page"])
    print("Similarity:", result["score"])
    print("Passage:", result["text"][:700])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]


--- Result 1 ---
Unit: Unit3
PDF page: 75
Similarity: 0.8
Passage: Backup and Archive Backup Granularity Backup granularity levels

--- Result 2 ---
Unit: Unit3
PDF page: 76
Similarity: 0.781
Passage: Backup and Archive Backup Granularity ✔Full backup is a backup of the complete data on the production volumes. ✔A full backup copy is created by copying the data in the production volumes to a backup storage device. ✔It provides a faster recovery but requires more storage space and also takes more time to back up. ✔Incremental backup copies the data that has changed since the last full or incremental backup, whichever has occurred more recently.

--- Result 3 ---
Unit: Unit3
PDF page: 74
Similarity: 0.716
Passage: Backup and Archive Backup Granularity ✔Backup granularity depends on business needs and the required RTO/RPO. ✔Based on the granularity, backups can be categorized as full, incremental and accumulative (differential). ✔Most organizations use a combination of these three backup 

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading Qwen for answer generation...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Qwen loaded successfully!")

Loading Qwen for answer generation...


`torch_dtype` is deprecated! Use `dtype` instead!


Qwen loaded successfully!


In [5]:
question = "Explain backup granularity"
marks = 10

results = retrieve_ism(question, top_k=3)

context = "\n\n".join(
    f"[Source: {r['unit']}, PDF page {r['page']}]\n{r['text']}"
    for r in results
)

prompt = f"""
You are an Information Storage Technology Management exam assistant.

Answer the following question for {marks} marks.
Use only the retrieved source passages below.
If the sources do not contain enough information, clearly say so.
Do not invent facts.
Cite supporting sources using the unit and PDF page.

Question: {question}

Retrieved passages:
{context}

Write a structured, exam-style answer with an introduction,
numbered points, and a conclusion where appropriate.
"""

messages = [
    {
        "role": "system",
        "content": "You provide clear, source-grounded ISM exam answers."
    },
    {"role": "user", "content": prompt}
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=3000
).to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=600,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

answer = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("QUESTION:", question)
print("MARKS:", marks)
print("\nANSWER:\n")
print(answer)
print("\nRETRIEVED SOURCES:")
for r in results:
    print(f"- {r['unit']}, PDF page {r['page']}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QUESTION: Explain backup granularity
MARKS: 10

ANSWER:

### Introduction:

Backup granularity refers to the level of detail captured during the backup process. It determines how much data is backed up at each point in time and affects the efficiency and cost-effectiveness of the backup strategy. This concept is crucial for ensuring data integrity while minimizing storage costs and improving recovery times.

### Points:

#### Full Backup:
- **Definition**: A full backup captures all data from the production volume at a specific point in time.
- **Purpose**: Provides a comprehensive snapshot of the entire system's state, which is essential for restoring the most recent version of the database or application.
- **Advantages**:
  - Faster recovery because it includes all necessary data.
  - Requires less frequent updates compared to incremental backups.
- **Disadvantages**:
  - Higher storage consumption due to the need to store the entire dataset.
  - Longer backup and restore times.

##

In [6]:
%pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 90.1 MB/s eta 0:00:00


In [7]:
%%writefile app.py
import streamlit as st

st.set_page_config(
    page_title="ISM AI Study Assistant",
    page_icon="📚",
    layout="wide"
)

st.title("📚 ISM AI Study Assistant")
st.caption(
    "Information Storage Technology Management — "
    "Retrieval-Augmented Generation (RAG) prototype"
)

st.info(
    "Ask a question from your ISM units. "
    "The system retrieves relevant PDF passages and generates "
    "an exam-style answer."
)

question = st.text_area(
    "Enter your ISM question",
    placeholder="Example: Explain backup granularity"
)

marks = st.selectbox(
    "Select the expected answer length",
    [2, 5, 10, 15, 20],
    index=2
)

if st.button("Generate Answer", type="primary"):
    if not question.strip():
        st.warning("Please enter a question.")
    else:
        st.session_state["ism_question"] = question
        st.session_state["ism_marks"] = marks
        st.info(
            "Your question has been captured. "
            "The next integration step will connect this interface "
            "to the RAG retrieval and answer-generation functions."
        )

st.divider()
st.subheader("Project Features")
st.markdown("""
- Four-unit ISM PDF knowledge base
- Semantic retrieval of relevant passages
- Exam-style answers for 2, 5, 10, 15 and 20 marks
- Unit and PDF-page source references
- Fine-tuning planned as a separate enhancement
""")

Writing app.py


In [8]:
%%writefile app.py
import json
import os
import streamlit as st
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForCausalLM

st.set_page_config(
    page_title="ISM AI Study Assistant",
    page_icon="📚",
    layout="wide"
)

@st.cache_resource
def load_resources():
    with open("/content/chunks.json", "r", encoding="utf-8") as f:
        chunks = json.load(f)

    embedder = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2"
    )
    texts = [c.get("text", "") for c in chunks]
    embeddings = embedder.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    model_name = "Qwen/Qwen2.5-1.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available()
        else torch.float32,
        device_map="auto"
    )

    return chunks, embedder, embeddings, tokenizer, model

st.title("📚 ISM AI Study Assistant")
st.caption("Information Storage Technology Management • RAG Prototype")

if not os.path.exists("/content/chunks.json"):
    st.error("chunks.json is missing from the Colab runtime.")
    st.stop()

question = st.text_area(
    "Enter your ISM question",
    placeholder="Example: Explain backup granularity"
)
marks = st.selectbox("Answer length", [2, 5, 10, 15, 20], index=2)

if st.button("Generate Answer", type="primary"):
    if not question.strip():
        st.warning("Please enter a question.")
        st.stop()

    with st.spinner("Loading the knowledge base and model..."):
        chunks, embedder, embeddings, tokenizer, model = load_resources()

    with st.spinner("Retrieving passages and generating your answer..."):
        query_vector = embedder.encode(
            [question],
            normalize_embeddings=True
        )
        scores = cosine_similarity(query_vector, embeddings)[0]
        best = scores.argsort()[-3:][::-1]

        results = [chunks[i] for i in best]
        context = "\n\n".join(
            f"[Source: {c.get('unit', 'Unknown')}, "
            f"PDF page {c.get('page_number', 'Unknown')}]\n"
            f"{c.get('text', '')}"
            for c in results
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "You are an Information Storage Technology Management "
                    "exam assistant. Use the provided sources as evidence. "
                    "Do not invent facts. If evidence is insufficient, "
                    "say so. Cite source unit and PDF page."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Answer this question for {marks} marks: {question}\n\n"
                    f"Use the retrieved passages below:\n{context}\n\n"
                    "Write a clear, structured exam-style answer."
                )
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=3000
        ).to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens= min(800, marks * 35),
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        answer = tokenizer.decode(
            output[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

    st.subheader(f"Answer ({marks} marks)")
    st.write(answer)

    st.subheader("Retrieved sources")
    for c in results:
        st.markdown(
            f"- **{c.get('unit', 'Unknown')}**, "
            f"PDF page **{c.get('page_number', 'Unknown')}**"
        )
        with st.expander("View retrieved passage"):
            st.write(c.get("text", ""))

st.divider()
st.caption(
    "Academic prototype. Verify generated answers against the original PDFs. "
    "Fine-tuning has not yet been completed."
)

Overwriting app.py


In [9]:
%pip install -q streamlit sentence-transformers scikit-learn

In [10]:
!streamlit run /content/app.py --server.port 8501 --server.headless true > /content/streamlit.log 2>&1 &

In [11]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸

In [13]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

In [15]:
!cat /content/streamlit.log



2026-10-18 19:21:39.411 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.187.240.160:8501

I0000 00:00:1792351829.638164   31158 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
  Stopping...


In [16]:
!pkill -f "streamlit run" || true

^C


In [17]:
!nohup streamlit run /content/app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.headless true \
  > /content/streamlit.log 2>&1 &

In [18]:
!sleep 5
!curl -I http://127.0.0.1:8501

HTTP/1.1 200 OK
date: Sun, 18 Oct 2026 19:45:46 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 6467
last-modified: Sun, 18 Oct 2026 19:13:24 GMT
etag: "ad3d1da9c07746d4624dd41fa81877eb"
cache-control: no-cache



In [19]:
import subprocess
import time
import re

# Start a new Cloudflare tunnel to your Streamlit app
process = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

print("Starting Cloudflare tunnel... Please wait.\n")

public_url = None
start_time = time.time()

while time.time() - start_time < 60:
    line = process.stdout.readline()

    if not line:
        if process.poll() is not None:
            break
        time.sleep(0.2)
        continue

    print(line.rstrip())

    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        print("\n" + "=" * 55)
        print("YOUR ISM AI STUDY ASSISTANT LINK:")
        print(public_url)
        print("=" * 55)
        break

if public_url is None:
    print("\nNo public link detected yet.")
    print("Check whether the Streamlit server is running on port 8501.")

Starting Cloudflare tunnel... Please wait.

2026-10-18T19:48:18Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-18T19:48:18Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-18T19:48:23Z INF +--------------------------------------------------------------------------------------------+
2026-10-18T19:48:23Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-18T19:48:23Z INF |  https://l